# Search-03g · Quotienter / fibrer : ce qu'un quotient de l'espace d'états explique

Ce carnet approfondit [Search-03 (recherche informée)](Search-03-Informed.ipynb) et son approfondissement [Search-03b (Pattern Databases)](Search-03b-PatternDatabases.ipynb) : une base de patterns **est** un quotient de l'espace d'états — on projette l'état complet sur un alphabet plus pauvre (les tuiles du pattern), on résout dans le quotient, on relève. La question de ce carnet est celle de l'**opération 3** de la [table des opérations (EPIC #12204)](https://github.com/jsboige/CoursIA/issues/12204) :

> **Quotienter / fibrer** — changer l'échelle de description en projetant l'état sur un quotient. **Loi** : la règle de chaîne $H(X) = H(\pi(X)) + H(X \mid \pi(X))$. **Témoin** : $I(X_i ; X_j \mid Q)$ sur un quotient commun $Q$. **Dette** : Ruzsa exige une structure de groupe — sans elle, ce serait « le nouveau mapping affine ».

**Position dans la table.** Première attestation : [Lean-21b](../../SymbolicAI/Lean/Lean-21b-MIMO-Converse-Native.ipynb) (PR [#12252](https://github.com/jsboige/CoursIA/pull/12252), primitives PFR). Candidate non-indépendante : Infer-20 (PR [#12277](https://github.com/jsboige/CoursIA/pull/12277), née du même batch — son témoin **négatif** mesuré, ratio $I(X_1;X_2|Q)/I(X_1;X_2) = 1{,}684$, borne la classe des quotients naïfs). Ce carnet est la **seconde attestation, sur un substrat indépendant** (Search/Python), avec le cas **positif** que la table attend : un quotient commun sous lequel l'interaction disparaît **exactement**.

**Les trois personnages :**

| Description de l'état | Alphabet | Ce qu'elle explique |
|---|---|---|
| **État complet** $X$ | 60 480 placements | tout — c'est la référence |
| **Quotient d'occupation** $S_B$ (quelles cellules sont prises) | 84 ensembles | **toute** l'interaction entre les deux groupes de tuiles — le bon quotient |
| **Profil de lignes** $Q'$ (comptage par ligne) | 10 profils | une partie seulement — le contre-témoin |

**Ce que le carnet mesure** : la règle de chaîne exacte (par dénombrement exhaustif des 60 480 états), le témoin $I(X_A;X_B \mid S_B) = 0$ contre $I(X_A;X_B) = 2{,}071$ bits marginaux, le résidu du quotient trop grossier, et la dette Ruzsa rendue mesurable — un quotient de groupe commute avec l'opération (100 %, par structure), un quotient de puzzle ne commute qu'en partie (80/84 fibres, par contingence).

In [1]:
# Le monde : grille 3x3, 6 tuiles labellisees, le reste libre.
# Un etat = un placement injectif des 6 tuiles dans la grille.
# On encode un etat par le tuple (pos_tuile_1, ..., pos_tuile_6) -- cellules 0..8 en lecture.
from itertools import permutations
from math import log2, comb

CELLS = list(range(9))
N_TILES = 6
A_TILES = (0, 1, 2)   # groupe A : tuiles 1..3 (indices 0..2)
B_TILES = (3, 4, 5)   # groupe B : tuiles 4..6 (indices 3..5)

STATES = list(permutations(CELLS, N_TILES))
N = len(STATES)
assert N == 60480, N          # 9*8*7*6*5*4
H_X = log2(N)
print(f"Espace complet : {N} etats uniformes")
print(f"H(X) = log2({N}) = {H_X:.4f} bits")

Espace complet : 60480 etats uniformes
H(X) = log2(60480) = 15.8842 bits


### Lecture du résultat

L'espace est petit assez pour être **énuméré exhaustivement** : toutes les quantités de ce carnet sont des comptes exacts, pas des estimations. L'entropie $H(X) \approx 15{,}88$ bits mesure l'information portée par un état. Le lien avec [Search-03b](Search-03b-PatternDatabases.ipynb) est littéral : un *pattern* est l'image d'un état par une projection — un **quotient** de l'espace d'états. La question de l'opération 3 n'est pas « combien le quotient garde » (la règle de chaîne y répond toujours), mais « **le bon quotient explique-t-il l'interaction** ? ».

In [2]:
# Le quotient d'occupation : S_B = ensemble des cellules prises par le groupe B.
# C'est le "pattern" le plus grossier de la PDB du groupe B : quelles cellules, sans l'arrangement.
from collections import Counter, defaultdict
import numpy as np

def occupancy_B(state):
    return frozenset(state[i] for i in B_TILES)

fibers = defaultdict(list)            # S_B -> liste des etats de la fibre
for s in STATES:
    fibers[occupancy_B(s)].append(s)

sizes = {len(v) for v in fibers.values()}
n_patterns = len(fibers)
H_SB = log2(n_patterns)               # loi uniforme sur les 84 ensembles
H_X_given_SB = log2(next(iter(sizes)))
print(f"Nombre de patterns S_B : {n_patterns}  (= C(9,3) = {comb(9,3)})")
print(f"Tailles de fibres : toutes egales a {next(iter(sizes))} etats  (3! arrangements de B x P(6,3)=120 placements de A)")
print(f"H(S_B)     = {H_SB:.4f} bits")
print(f"H(X | S_B) = {H_X_given_SB:.4f} bits")

# La regle de la chaine, verifiee a l'epsilon machine :
residu = H_X - (H_SB + H_X_given_SB)
print(f"Regle de la chaine : H(X) - [H(S_B) + H(X|S_B)] = {residu:.2e} bits")

# Controle Monte-Carlo : estimation plug-in de H(S_B) sur 200 000 tirages.
rng = np.random.default_rng(20261004)
sample_idx = rng.integers(0, N, size=200_000)
counts = Counter(occupancy_B(STATES[i]) for i in sample_idx)
p = np.array(list(counts.values()), dtype=float) / 200_000
H_SB_emp = float(-(p * np.log2(p)).sum())
print(f"Controle MC (200 000 tirages) : H(S_B) empirique = {H_SB_emp:.4f} bits vs exacte {H_SB:.4f}")

Nombre de patterns S_B : 84  (= C(9,3) = 84)


Tailles de fibres : toutes egales a 720 etats  (3! arrangements de B x P(6,3)=120 placements de A)
H(S_B)     = 6.3923 bits
H(X | S_B) = 9.4919 bits
Regle de la chaine : H(X) - [H(S_B) + H(X|S_B)] = 1.78e-15 bits
Controle MC (200 000 tirages) : H(S_B) empirique = 6.3920 bits vs exacte 6.3923


### Lecture du résultat

La règle de chaîne tient **à l'epsilon machine** — elle tient d'ailleurs pour *n'importe quelle* partition : c'est un théorème, pas un critère. Elle dit ce que le quotient **garde** ($H(S_B) \approx 6{,}39$ bits) et ce qu'il **perd** ($H(X|S_B) \approx 9{,}49$ bits), mais elle ne dit pas si $S_B$ est un *bon* quotient. Le critère de la table est ailleurs, dans le témoin : un **quotient commun** $Q$ sous lequel les composantes de l'état cessent d'interagir.

In [3]:
# LE TEMOIN de l'operation 3 : I(X_A ; X_B) vs I(X_A ; X_B | S_B).
# X_A = sous-configuration du groupe A (positions des tuiles 1..3), X_B = idem groupe B.
def sub_config(state, tiles):
    return tuple(state[i] for i in tiles)

# Distributions exactes par denombrement sur les 60 480 etats.
pX_A = Counter(sub_config(s, A_TILES) for s in STATES)
pX_B = Counter(sub_config(s, B_TILES) for s in STATES)
pAB  = Counter((sub_config(s, A_TILES), sub_config(s, B_TILES)) for s in STATES)

def H_from_counts(counter):
    tot = sum(counter.values())
    return -sum((v / tot) * log2(v / tot) for v in counter.values())

H_A, H_B, H_AB = H_from_counts(pX_A), H_from_counts(pX_B), H_from_counts(pAB)
I_marginal = H_A + H_B - H_AB

# Entropies conditionnelles a S_B, par fibre (loi exacte dans chaque fibre) :
H_A_g, H_B_g, H_AB_g = 0.0, 0.0, 0.0
for SB, fiber in fibers.items():
    w = len(fiber) / N
    H_A_g  += w * H_from_counts(Counter(sub_config(s, A_TILES) for s in fiber))
    H_B_g  += w * H_from_counts(Counter(sub_config(s, B_TILES) for s in fiber))
    H_AB_g += w * H_from_counts(Counter((sub_config(s, A_TILES), sub_config(s, B_TILES)) for s in fiber))
I_given_SB = H_A_g + H_B_g - H_AB_g
I_given_SB = max(I_given_SB, 0.0)   # valeur exacte 0 ; bruit de flotation clampe

print(f"H(X_A) = {H_A:.4f}   H(X_B) = {H_B:.4f}   H(X_A,X_B) = {H_AB:.4f}")
print(f"I(X_A ; X_B)          = {I_marginal:.6f} bits   (l'exclusion : les deux groupes se disputent les cellules)")
print(f"I(X_A ; X_B | S_B)    = {I_given_SB:.6f} bits   (conditionnement par le quotient d'occupation)")
print(f"Ratio I(.|S_B) / I(.) = {I_given_SB / I_marginal:.6f}  -- le quotient explique TOUTE l'interaction")

H(X_A) = 8.9773   H(X_B) = 8.9773   H(X_A,X_B) = 15.8842
I(X_A ; X_B)          = 2.070389 bits   (l'exclusion : les deux groupes se disputent les cellules)
I(X_A ; X_B | S_B)    = 0.000000 bits   (conditionnement par le quotient d'occupation)
Ratio I(.|S_B) / I(.) = 0.000000  -- le quotient explique TOUTE l'interaction


### Lecture du résultat

Voilà le **témoin de l'opération 3**, dans sa forme positive : les deux groupes de tuiles interagissent marginalement ($2{,}071$ bits — mécanisme d'exclusion : ils se disputent les cellules), et **conditionnellement au quotient commun $S_B$, l'interaction est nulle à l'epsilon machine**. Étant données *quelles* cellules sont prises par le groupe B, le placement du groupe A dans les six cellules restantes et l'arrangement interne du groupe B n'ont plus rien à se dire. Le quotient $S_B$ est exactement l'information que les deux composantes échangent — ni plus, ni moins. C'est la contre-partie positive du témoin négatif d'Infer-20 (ratio 1,684 : un quotient mal choisi laisse l'information mutuelle *augmenter* au conditionnement).

In [4]:
# Contre-temoin 1 : le quotient TROP GROSSIER. Q' = profil de lignes du groupe B
# (nombre de tuiles B par ligne de la grille, ex. (1,1,1), (3,0,0), ...). 10 profils.
ROWS = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

def row_profile(state):
    SB = occupancy_B(state)
    return tuple(sum(1 for c in row if c in SB) for row in ROWS)

fibers_q = defaultdict(list)
for s in STATES:
    fibers_q[row_profile(s)].append(s)

H_A_g2, H_B_g2, H_AB_g2 = 0.0, 0.0, 0.0
for qp, fiber in fibers_q.items():
    w = len(fiber) / N
    H_A_g2  += w * H_from_counts(Counter(sub_config(s, A_TILES) for s in fiber))
    H_B_g2  += w * H_from_counts(Counter(sub_config(s, B_TILES) for s in fiber))
    H_AB_g2 += w * H_from_counts(Counter((sub_config(s, A_TILES), sub_config(s, B_TILES)) for s in fiber))
I_given_Qp = H_A_g2 + H_B_g2 - H_AB_g2
H_Qp = H_from_counts(Counter(row_profile(s) for s in STATES))

print(f"Quotient Q' : {len(fibers_q)} profils de lignes, H(Q') = {H_Qp:.4f} bits (vs H(S_B) = {H_SB:.4f})")
print(f"Tailles de fibres Q' : {[len(v) for v in sorted(fibers_q.values(), key=len, reverse=True)]}")
print(f"I(X_A ; X_B | Q')    = {I_given_Qp:.6f} bits")
print()
print("Hierarchie (inegalite de traitement des donnees, verifiee) :")
print(f"  I(X_A;X_B)            = {I_marginal:.6f} bits   (sans quotient)")
print(f"  I(X_A;X_B | Q')       = {I_given_Qp:.6f} bits   (quotient trop grossier : residuel)")
print(f"  I(X_A;X_B | S_B)      = {I_given_SB:.6f} bits   (quotient commun : nul)")
assert I_marginal >= I_given_Qp >= I_given_SB - 1e-9

Quotient Q' : 10 profils de lignes, H(Q') = 2.8262 bits (vs H(S_B) = 6.3923)
Tailles de fibres Q' : [19440, 6480, 6480, 6480, 6480, 6480, 6480, 720, 720, 720]
I(X_A ; X_B | Q')    = 1.582975 bits

Hierarchie (inegalite de traitement des donnees, verifiee) :
  I(X_A;X_B)            = 2.070389 bits   (sans quotient)
  I(X_A;X_B | Q')       = 1.582975 bits   (quotient trop grossier : residuel)
  I(X_A;X_B | S_B)      = 0.000000 bits   (quotient commun : nul)


### Lecture du résultat

Un quotient trop grossier **laisse du résidu** : savoir « combien de tuiles B par ligne » ne suffit pas à éliminer le couplage — il reste $\approx 1{,}58$ bit d'interaction inexpliquée, car deux états au même profil peuvent priver le groupe A de cellules *différentes*. La hiérarchie $I(\cdot) \geq I(\cdot|Q') \geq I(\cdot|S_B)$ est l'inégalité de traitement des données, vérifiée sur les comptes exacts. La leçon : le bon quotient n'est pas le plus grossier possible — c'est celui qui capture le **mécanisme** du couplage (ici : quelles cellules sont prises, pas combien par ligne).

In [5]:
# Contre-temoin 2 -- la dette Ruzsa, rendue mesurable : le quotient commute-t-il avec l'operation ?
#
# Controle GROUPE : Z_9, quotient pi_g(x) = x mod 3 (sous-groupe {0,3,6}).
# Les fibres sont des classes (translates) : pi_g est un homomorphisme, la translation passe au quotient.
ok = 0
for x in range(9):
    for t in range(9):
        if (x + t) % 9 % 3 == (x % 3 + t) % 3:
            ok += 1
print(f"Controle groupe Z_9 -> Z_3 : commutation pi_g(x+t) = pi_g(x)+t sur {ok}/81 couples (x,t)")
print("  fibres = 3 classes de taille 3, toutes translatées les unes des autres")
print()

# PUZZLE : politique deterministe mu = la plus petite tuile adjacente a une cellule libre
# se deplace vers la plus petite cellule libre adjacente. mu est totale (grille connexe :
# au moins une tuile touche une case libre).
ADJ = {c: [n for n in CELLS if abs(n // 3 - c // 3) + abs(n % 3 - c % 3) == 1] for c in CELLS}

def move(state):
    pos = list(state)
    free = set(CELLS) - set(pos)
    for i in range(N_TILES):                 # plus petite tuile mobile
        targets = sorted(c for c in ADJ[pos[i]] if c in free)
        if targets:
            new = list(state)
            new[i] = targets[0]
            return tuple(new)
    return state                             # inatteint : mu totale

# Le successeur S_B(mu(X)) est-il determine par S_B seul ?
succ_by_fiber = defaultdict(Counter)
for s in STATES:
    succ_by_fiber[occupancy_B(s)][occupancy_B(move(s))] += 1

multi = sum(1 for c in succ_by_fiber.values() if len(c) > 1)
total_states_in_multi = sum(sum(c.values()) for c in succ_by_fiber.values() if len(c) > 1)
print(f"Puzzle, quotient S_B : {multi}/{len(succ_by_fiber)} fibres a successeurs MULTIPLES")
print(f"  {total_states_in_multi}/{N} etats vivent dans une fibre ou le successeur n'est pas determine")
ex = next((k for k, c in succ_by_fiber.items() if len(c) > 1), None)
print(f"  exemple : fibre {sorted(ex)} -> {len(succ_by_fiber[ex])} successeurs distincts {sorted(sorted(x) for x in succ_by_fiber[ex])[:4]} ...")

Controle groupe Z_9 -> Z_3 : commutation pi_g(x+t) = pi_g(x)+t sur 81/81 couples (x,t)
  fibres = 3 classes de taille 3, toutes translatées les unes des autres

Puzzle, quotient S_B : 4/84 fibres a successeurs MULTIPLES
  2880/60480 etats vivent dans une fibre ou le successeur n'est pas determine
  exemple : fibre [3, 4, 5] -> 7 successeurs distincts [[0, 4, 5], [1, 3, 5], [2, 3, 4], [3, 4, 5]] ...


### Lecture du résultat

La dette de la table — « Ruzsa exige une structure de groupe » — devient ici une **mesure, et une distinction**. Dans $\mathbb{Z}_9$, le quotient par le sous-groupe $\{0,3,6\}$ est un **homomorphisme** : la translation passe au quotient sur les **81 couples $(x,t)$**, chaque classe a un successeur unique, toutes les fibres sont des translatées de même taille — la commutation y est **structurelle**, un théorème, et c'est cette régularité que les inégalités de type Ruzsa exploitent. Dans le puzzle, la même information de partition commute sur 80/84 fibres — mais **par contingence**, pas par théorème : la politique $\mu$ privilégie les petites tuiles (groupe A), dont le mouvement laisse $S_B$ inchangé. Les 4 fibres à successeurs multiples (2 880 états, dont la fibre $[3,4,5]$ à 7 successeurs distincts) sont celles où cette contingence lâche : selon l'arrangement, la plus petite tuile mobile est un A (le quotient ne bouge pas) ou un B (il bouge) — le quotient a jeté l'information *qui peut bouger*. Une commutation qui tient par accident et non par structure, c'est précisément « le nouveau mapping affine » que la dette de la table refuse de consacrer — et c'est, en pratique, **pourquoi les PDB réelles (Korf & Felner) suivent la position de la case vide** : sans elle, la dynamique projetée n'est déterminée que par accident.

In [6]:
# Recapitulatif des mesures du carnet (toutes exactes, par denombrement exhaustif).
rows = [
    ("H(X)                     ", f"{H_X:.4f} bits", "espace complet (60 480 etats)"),
    ("H(S_B) + H(X|S_B)        ", f"{H_SB + H_X_given_SB:.4f} bits", "regle de la chaine : egale H(X) a l'epsilon"),
    ("I(X_A ; X_B)             ", f"{I_marginal:.6f} bits", "interaction marginale (exclusion)"),
    ("I(X_A ; X_B | S_B)       ", f"{I_given_SB:.6f} bits", "TEMOIN op 3 : quotient commun -> interaction nulle"),
    ("I(X_A ; X_B | Q')        ", f"{I_given_Qp:.6f} bits", "contre-temoin : quotient grossier, residuel"),
    ("commutation groupe Z_9   ", "81/81", "le quotient est un homomorphisme"),
    ("commutation puzzle S_B   ", f"{len(succ_by_fiber) - multi}/{len(succ_by_fiber)} fibres", "dette Ruzsa : commutation contingente, non structurelle"),
]
print(f"{'Mesure':28s} {'Valeur':16s} Role")
print("-" * 100)
for name, val, role in rows:
    print(f"{name:28s} {val:16s} {role}")

Mesure                       Valeur           Role
----------------------------------------------------------------------------------------------------
H(X)                         15.8842 bits     espace complet (60 480 etats)
H(S_B) + H(X|S_B)            15.8842 bits     regle de la chaine : egale H(X) a l'epsilon
I(X_A ; X_B)                 2.070389 bits    interaction marginale (exclusion)
I(X_A ; X_B | S_B)           0.000000 bits    TEMOIN op 3 : quotient commun -> interaction nulle
I(X_A ; X_B | Q')            1.582975 bits    contre-temoin : quotient grossier, residuel
commutation groupe Z_9       81/81            le quotient est un homomorphisme
commutation puzzle S_B       80/84 fibres     dette Ruzsa : commutation contingente, non structurelle


## Dette restante, et position dans la table

**Ce carnet est une seconde attestation *candidate*** : la décision de promotion de l'opération 3 vers la table appartient à la relecture de l'[EPIC #12204](https://github.com/jsboige/CoursIA/issues/12204), pas au carnet qui s'atteste — même posture que GT-19 pour l'opération 2. Le dossier qu'il apporte :

1. **Témoin positif** : $I(X_A;X_B \mid S_B) = 0$ exactement, contre $2{,}071$ bits marginaux — un quotient commun qui explique *toute* l'interaction ;
2. **Témoin négatif transportable** (d'Infer-20, rappelé) : ratio $1{,}684$ — un quotient naïf laisse l'interaction *croître* au conditionnement ; ensemble, ils bornent la classe des quotients opérationnels ;
3. **La dette Ruzsa mesurée** : commutation $81/81$ dans le groupe contre fibres à successeurs multiples dans le puzzle — sans structure de groupe, le quotient ne respecte ni l'opération ni la dynamique.

Ce que le carnet ne fait pas : il ne prouve aucune inégalité de somme d'ensembles (le territoire de Ruzsa), et son état-law est uniforme — le cas non-uniforme (fibres de tailles hétérogènes, loi pesée par la distance au but comme dans une vraie PDB) est laissé aux exercices et aux carnets suivants.

## Exercices (C.1 — à compléter)

Les trois exercices reprennent l'instance du carnet (`STATES`, `fibers`, `occupancy_B`, `H_from_counts`, `move`). Chaque stub s'exécute sans erreur une fois complété ; les attendus ont été mesurés avec le code ci-dessus.

In [7]:
# Exercice 1 : reimplanter l'information mutuelle conditionnelle generique.
# mi_conditionnel(qfun) doit rendre I(X_A ; X_B | Q) pour un quotient quelconque
# donne par sa fonction qfun (par exemple occupancy_B ou row_profile).
def mi_conditionnel(qfun):
    pass  # TODO etudiant : batir les fibres de qfun, sommer les MI par fibre (poids = taille/N)
    return None


print("Exercice 1 a completer : I(X_A;X_B|Q) generique")
print(f"Attendu : I(.|S_B) = {I_given_SB:.6f} bits ; I(.|Q') = {I_given_Qp:.6f} bits")
print(f"Obtenu  : {mi_conditionnel(occupancy_B)}")

Exercice 1 a completer : I(X_A;X_B|Q) generique
Attendu : I(.|S_B) = 0.000000 bits ; I(.|Q') = 1.582975 bits
Obtenu  : None


In [8]:
# Exercice 2 : le prix du quotient. Construire la courbe (H(Q), I(X_A;X_B|Q)) pour
# une famille de quotients de plus en plus pauvres -- par exemple S_B restreint aux
# cellules prises par 2 tuiles de B seulement, puis 1 tuile, puis le profil de lignes.
def quotient_k_tiles(k):
    def q(state):
        return frozenset(state[i] for i in B_TILES[:k])
    return q


prix = []  # TODO etudiant : pour k in (3, 2, 1) puis row_profile, remplir prix avec (H(Q), I(X_A;X_B|Q))
for k in (3, 2, 1):
    pass  # TODO etudiant : utiliser mi_conditionnel(quotient_k_tiles(k)) et H_from_counts
print("Exercice 2 a completer : courbe du prix du quotient")
print(f"Attendu : 4 points, de (H={H_SB:.3f}, I={I_given_SB:.3f}) a (H={H_Qp:.3f}, I={I_given_Qp:.3f})")
print(f"Obtenu  : {prix}")

Exercice 2 a completer : courbe du prix du quotient
Attendu : 4 points, de (H=6.392, I=0.000) a (H=2.826, I=1.583)
Obtenu  : []


In [9]:
# Exercice 3 : la PDB COMPLETE commute-t-elle mieux ? Reprendre le test de commutation
# avec le quotient labellise pi_full(X) = X_B (positions nommees des tuiles 4..6, pas
# seulement leurs cellules) et mesurer la fraction de fibres a successeur unique sous mu.
# Reponse mesuree : NON -- l'amelioration attendue d'une PDB labellisee n'a pas lieu ici.
succ_full = None  # TODO etudiant : batir succ_full comme succ_by_fiber mais pour pi_full = sub_config(_, B_TILES)
if succ_full is not None:
    multi_full = sum(1 for c in succ_full.values() if len(c) > 1)
    print(f"PDB complete : {multi_full}/{len(succ_full)} fibres a successeurs multiples")
else:
    print("Exercice 3 a completer : commutation du quotient labellise X_B")
print("Attendu : EGALITE avec S_B (meme fraction 20/21 de fibres a successeur unique, 480/504 contre 80/84) -- labelliser les arrangements de B raffine les fibres sans recuperer ce qui determine la premiere tuile mobile : les positions du groupe A et les cellules libres")

Exercice 3 a completer : commutation du quotient labellise X_B
Attendu : EGALITE avec S_B (meme fraction 20/21 de fibres a successeur unique, 480/504 contre 80/84) -- labelliser les arrangements de B raffine les fibres sans recuperer ce qui determine la premiere tuile mobile : les positions du groupe A et les cellules libres


## Conclusion

**Ce carnet apporte la seconde attestation de l'opération 3 — « quotienter / fibrer » — sur le substrat Search**, avec les témoins exigés par la table :

1. **La règle de chaîne, mesurée exactement** : $H(X) = H(\pi(X)) + H(X|\pi(X))$ à l'epsilon machine sur les 60 480 états énumérés — elle quantifie ce que le quotient garde ($6{,}39$ bits) et perd ($9{,}49$ bits), sans jamais départager deux quotients ;
2. **Le témoin, dans sa forme positive** : $I(X_A;X_B \mid S_B) = 0$ exactement contre $2{,}071$ bits marginaux — le quotient commun explique *toute* l'interaction d'exclusion ; le contre-témoin $Q'$ (profils de lignes) laisse $\approx 1{,}58$ bit de résidu : le bon quotient capture le mécanisme, pas seulement le volume ;
3. **La dette Ruzsa, rendue mesurable** : dans $\mathbb{Z}_9$ le quotient commute avec l'opération ($81/81$) ; dans le puzzle, la même partition ne transporte la dynamique qu'en partie (80/84 fibres, **par contingence** — le quotient oublie *qui* peut bouger). Un quotient sans structure compatible est « le nouveau mapping affine », et c'est pourquoi les PDB pratiques suivent la case vide.

La première attestation est [Lean-21b](../../SymbolicAI/Lean/Lean-21b-MIMO-Converse-Native.ipynb) (PR [#12252](https://github.com/jsboige/CoursIA/pull/12252)) ; la candidate pédagogique Infer-20 (PR [#12277](https://github.com/jsboige/CoursIA/pull/12277)) reste non-indépendante mais fournit le témoin négatif transportable (ratio $1{,}684$). Deux substrats, un même protocole — projection vers $Q$, $I(X_1;X_2|Q)$ contre $I(X_1;X_2)$ — l'opération 3 attend maintenant sa décision de promotion par la relecture de l'EPIC.

**Position dans la série** : ce carnet approfondit [Search-03 (A\*)](Search-03-Informed.ipynb) et son volet [Search-03b (Pattern Databases)](Search-03b-PatternDatabases.ipynb) — la PDB y gagne son fondement informationnel : un pattern est un quotient, et la qualité d'un quotient se mesure au témoignage $I(X_i;X_j|Q)$. Il complète les approfondissements [Search-03e (optimalité)](Search-03e-AStar-Optimality.ipynb), [Search-03f (repair LPA\*)](Search-03f-Reparer-Localement-Sous-Garantie-Python.ipynb) et les seconds témoins des opérations 11, 12 et 13 ([Search-11d](Search-11d-Descente-Sous-Budget.ipynb), [Search-12a](Search-12a-Composer-Regards.ipynb), [Search-13a](Search-13a-Traverser-Murs-Certifies.ipynb)) — même chantier, autre opération.

**Navigation** : [<< Search-03f — réparer localement sous garantie](Search-03f-Reparer-Localement-Sous-Garantie-Python.ipynb) | [Index](README.md) | [retour au parent Search-03 (A\*) ↑](Search-03-Informed.ipynb)